# **Mount Drive**

# **Load Data**

In [28]:
import pandas as pd
ffail = '<DATA_ROOT>/crop_failure_modified/crops_failure_0595_20240221.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])

cond1 = df_fail['Planted Acres'] > 0
df_fail = df_fail[cond1]
df_fail.loc[:,'fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']
df_fail = df_fail[df_fail.fail_share < 1]

cols = ['FIPS', 'Crop', 'Irrigation Practice', 'year', 'fail_share']

df_fail = df_fail[cols]
df_fail.groupby('Irrigation Practice').count()

,FIPS,Crop,year,fail_share
Irrigation Practice,,,,
ALL,217980,217980,217980,217980
I,70901,70901,70901,70901
N,162534,162534,162534,162534


In [29]:
import pandas as pd
import numpy as np
from scipy.signal import detrend

fpath_yield = '<DATA_ROOT>/Yield/crops_us_yield_agg_irig.csv'
dfy = pd.read_csv(fpath_yield)
df_grouped = dfy.groupby(['state_name','county_name','prodn_practice_desc','commodity_desc'])

def detrend_yield(yield_series):
  detrended_yield = np.zeros(len(yield_series))
  if len(yield_series) > 2:
    detrended_yield = detrend(yield_series)
  return pd.Series(detrended_yield, index=yield_series.index)

dfy['detrended_yield'] = df_grouped['value_yield'].transform(detrend_yield)
dfy['d_yield_standard'] = dfy['detrended_yield'] / (dfy['value_yield'] - dfy['detrended_yield'])

fpath_fips = '<DATA_ROOT>/Yield/yield_fips_code.csv'
dffipsy = pd.read_csv(fpath_fips).rename(columns={' county_code':'county_code'})
df_new = dffipsy.copy()
formatted_numbers = [str(n).zfill(3) for n in df_new['county_code'].values]
df_new['county_code'] = formatted_numbers
formatted_numbers = [str(n).zfill(2) for n in df_new['state_fips_code'].values]
df_new['state_fips_code'] = formatted_numbers
df_new['FIPS'] = df_new['state_fips_code'] + df_new['county_code']
dffipsy = df_new
dffipsy = dffipsy.drop(columns=['state_fips_code' , 'county_code'])
dffipsy

dfy = pd.merge(dfy,dffipsy,on=['state_name', 'county_name'])

cols = ['FIPS','commodity_desc','prodn_practice_desc','year','d_yield_standard']
df_yield = dfy[cols]
condy = df_yield.year > 2008
df_yield = df_yield[condy]
# df_yield
df_yield.groupby('prodn_practice_desc').count()

,FIPS,commodity_desc,year,d_yield_standard
prodn_practice_desc,,,,
ALL PRODUCTION PRACTICES,86700,86700,86700,86700
IRRIGATED,2946,2946,2946,2946
NON-IRRIGATED,3425,3425,3425,3425


In [30]:
from glob import glob
import pandas as pd
in_path = '<DATA_ROOT>/SVI_weightedCrop_Final/*.csv'
files = glob(in_path)

list_df = []
for f in files:
  crop = f.split('/')[-1].split('.')[0][4:]
  # print(crop)
  dfi = pd.read_csv(f).copy()
  dfi['EPL_PCI'] = dfi['EPL_PCI'].fillna(dfi['EPL_HBURD'])
  dfi = dfi.drop(columns=['EPL_HBURD' ,	'EPL_UNINSUR'])
  dfi['Crop'] = crop
  list_df.append(dfi)

df_svi = pd.concat(list_df,ignore_index=True)
df_svi = df_svi.set_index(['STCNTY','YEAR','Crop']).dropna(how='all').reset_index()
df_svi = df_svi.rename(columns={'STCNTY':'FIPS','YEAR':'year'})
df_svi

,FIPS,year,Crop,EPL_POV,EPL_UNEMP,EPL_PCI,EPL_NOHSDP,RPL_THEME1,EPL_AGE65,EPL_AGE17,...,EPL_MINRTY,EPL_LIMENG,RPL_THEME3,EPL_MUNIT,EPL_MOBILE,EPL_CROWD,EPL_NOVEH,EPL_GROUPQ,RPL_THEME4,RPL_THEMES
0,1009,2008,HAY,0.411000,0.269000,0.555000,0.623000,0.460000,0.394000,0.551000,...,0.154000,0.039000,0.002000,0.000000,0.947000,0.087000,0.174000,0.714000,0.458000,0.342000
1,1015,2008,HAY,0.440000,0.502000,0.430000,0.409000,0.431000,0.567000,0.293000,...,0.512000,0.149000,0.287000,0.435000,0.761000,0.050000,0.291000,0.000000,0.285000,0.322000
2,1043,2008,HAY,0.602000,0.376400,0.674400,0.718400,0.638000,0.541800,0.443200,...,0.069000,0.195600,0.069600,0.045800,0.916200,0.203400,0.471800,0.000000,0.270200,0.369200
3,1083,2008,HAY,0.562800,0.329280,0.555440,0.834680,0.610120,0.283880,0.512200,...,0.628960,0.331480,0.486560,0.012440,0.872000,0.522080,0.520000,0.039680,0.400400,0.496320
4,1089,2008,HAY,0.414111,0.390333,0.205222,0.235889,0.241000,0.192444,0.441667,...,0.557556,0.372444,0.469778,0.553111,0.706889,0.209778,0.168556,0.081111,0.310667,0.216556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279115,56037,2022,WHEAT,0.391589,0.622458,0.069057,0.258812,0.294302,0.679727,0.391632,...,0.118765,0.458488,0.118765,0.036466,0.961560,0.548027,0.150776,0.477634,0.462214,0.425930
279116,56039,2022,WHEAT,0.383900,0.050800,0.414700,0.239700,0.166200,0.976400,0.026500,...,0.200700,0.234300,0.200700,0.693800,0.688100,0.000000,0.681100,0.945700,0.805100,0.278600
279117,56041,2022,WHEAT,0.369025,0.571650,0.092250,0.220525,0.334900,0.420525,0.865150,...,0.220250,0.200225,0.220250,0.396775,0.970600,0.808525,0.265100,0.222375,0.624025,0.438500
279118,56043,2022,WHEAT,0.448187,0.361078,0.085784,0.274480,0.309528,0.786877,0.531011,...,0.143210,0.020884,0.143210,0.341739,0.805374,0.796464,0.405524,0.797343,0.839735,0.458316


In [31]:
import pandas as pd
import numpy as np
fpath_water = '<DATA_ROOT>/WaterUsage/WaterUse_crop_v3.csv'
dfw = pd.read_csv(fpath_water).rename(columns={'Year':'year'})
dfw = dfw[['FIPS','IR-WGWFr','IR-WSWFr','IR-WFrTo','IR-IrTot','year']]
dfw['IR-WGWFr'] = pd.to_numeric(dfw['IR-WGWFr'], errors='coerce')
dfw['IR-WSWFr'] = pd.to_numeric(dfw['IR-WSWFr'], errors='coerce')
dfw['IR-IrTot'] = pd.to_numeric(dfw['IR-IrTot'], errors='coerce')
dfw = dfw[dfw['IR-IrTot'] > 0]
dfw.loc[:,'wu_ratio'] = (dfw['IR-WGWFr'] > dfw['IR-WSWFr']).astype(int)
dfw.loc[:,'gwu_rate'] = dfw['IR-WGWFr'] / dfw['IR-IrTot']
dfw.loc[:,'swu_rate'] = dfw['IR-WSWFr'] / dfw['IR-IrTot']
formatted_numbers = [str(n).zfill(5) for n in dfw['FIPS'].values]
dfw['FIPS'] = formatted_numbers
dfw = dfw[['FIPS','year','gwu_rate','swu_rate']]
dfw['FIPS'] = dfw['FIPS'].astype(int)
dfw

,FIPS,year,gwu_rate,swu_rate
0,1001,2009,1.610738,0.255034
1,1003,2009,2.236618,0.428832
2,1005,2009,0.166667,0.498148
3,1007,2009,0.214286,0.428571
4,1009,2009,0.508475,0.779661
...,...,...,...,...
46609,56037,2023,0.416110,3.546957
46610,56039,2023,0.000000,6.989101
46611,56041,2023,0.573282,4.887083
46612,56043,2023,0.207130,6.698673


In [32]:
finsure = '<DATA_ROOT>/InsuranceData/InsuranceData.csv'
df_insure = pd.read_csv(finsure)
df_insure['FIPS'] = df_insure['State_Code'] * 1000 + df_insure['County_Code']
cols = ['FIPS', 'Commodity_Name', 'Year_Identifier','Cause_of_Loss_Code', 'Cause_of_Loss_Desc']
df_insure = df_insure[cols]
df_insure = df_insure.rename(columns={'Commodity_Name':'Crop','Year_Identifier':'year'})
cond1 = df_insure['Cause_of_Loss_Code'] != 'XX'
df_insure = df_insure[cond1]
df_insure['Cause_of_Loss_Code'] = df_insure['Cause_of_Loss_Code'].astype(int)
df_insure = df_insure.groupby(['FIPS','Crop','year','Cause_of_Loss_Code']).first().reset_index()
df_insure

<ipython-input-32-29954be7df5c>:2: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_insure = pd.read_csv(finsure)
<ipython-input-32-29954be7df5c>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_insure['Cause_of_Loss_Code'] = df_insure['Cause_of_Loss_Code'].astype(int)


,FIPS,Crop,year,Cause_of_Loss_Code,Cause_of_Loss_Desc
0,1001,CORN,2008,11,Drought
1,1001,CORN,2009,1,Decline in Price
2,1001,CORN,2009,11,Drought
3,1001,CORN,2009,12,Heat
4,1001,CORN,2009,31,Excess Moisture/Precipitation/Rain
...,...,...,...,...,...
400252,56045,WHEAT,2016,1,Decline in Price
400253,56045,WHEAT,2016,11,Drought
400254,56045,WHEAT,2016,42,Freeze
400255,56045,WHEAT,2017,1,Decline in Price


In [33]:
fcold = '<DATA_ROOT>/ClimateIndex/ColdSpell.csv'
df_cold = pd.read_csv(fcold)
cols = ['FIPS','Year','Thr-2','Thr-1','Thr0']
df_cold = df_cold[cols]
df_cold = df_cold.dropna(subset=['FIPS', 'Year'])
df_cold['FIPS'] = df_cold['FIPS'].astype(int)
df_cold['Year'] = df_cold['Year'].astype(int)
df_cold = df_cold.rename(columns={'Year':'year'})

In [34]:
fdrought = '<DATA_ROOT>/ClimateIndex/list_droughtFeatures.csv'
df_drought = pd.read_csv(fdrought)
df_drought = df_drought.dropna(subset=['Fips', 'Year'])
df_drought['Fips'] = df_drought['Fips'].astype(int)
df_drought['Year'] = df_drought['Year'].astype(int)
df_drought = df_drought.rename(columns={'Fips':'FIPS'})
df_drought = df_drought.rename(columns={'Year':'year'})

In [35]:
fprcp = '<DATA_ROOT>/ClimateIndex/prcp.csv'
df_prcp = pd.read_csv(fprcp)
df_prcp = df_prcp.dropna(subset=['FIPS', 'Year'])
df_prcp['FIPS'] = df_prcp['FIPS'].astype(int)
df_prcp['Year'] = df_prcp['Year'].astype(int)
df_prcp = df_prcp.drop(columns=['State','County'])
df_prcp = df_prcp.rename(columns={'Year':'year'})

In [36]:
fvpd = '<DATA_ROOT>/ClimateIndex/vpdfeatures.csv'
df_vpd = pd.read_csv(fvpd)
df_vpd = df_vpd.dropna(subset=['FIPS', 'Year'])
df_vpd['FIPS'] = df_vpd['FIPS'].astype(int)
df_vpd['Year'] = df_vpd['Year'].astype(int)
df_vpd = df_vpd.drop(columns=['State','County'])
df_vpd = df_vpd.rename(columns={'Year':'year'})

In [37]:
fwarm = '<DATA_ROOT>/ClimateIndex/WarmSpell.csv'
df_warm = pd.read_csv(fwarm)
df_warm = df_warm.dropna(subset=['FIPS', 'Year'])
df_warm['FIPS'] = df_warm['FIPS'].astype(int)
df_warm['Year'] = df_warm['Year'].astype(int)
df_warm = df_warm.drop(columns=['State','County'])
df_warm = df_warm.rename(columns={'Year':'year'})

In [38]:
# df_fail , df_yield , df_svi , dfw , df_insure

In [39]:
df_insure = df_insure.drop(columns=['Cause_of_Loss_Code'])
df_insure

,FIPS,Crop,year,Cause_of_Loss_Desc
0,1001,CORN,2008,Drought
1,1001,CORN,2009,Decline in Price
2,1001,CORN,2009,Drought
3,1001,CORN,2009,Heat
4,1001,CORN,2009,Excess Moisture/Precipitation/Rain
...,...,...,...,...
400252,56045,WHEAT,2016,Decline in Price
400253,56045,WHEAT,2016,Drought
400254,56045,WHEAT,2016,Freeze
400255,56045,WHEAT,2017,Decline in Price


In [40]:
df_svi = df_svi[['FIPS','Crop','year','RPL_THEMES']]
df_cold = df_cold[['FIPS', 'year','Thr-2']]
df_drought = df_drought[['FIPS', 'year','pdsi-2']] # 'spi1y-2'
df_prcp = df_prcp[['FIPS', 'year','dpi1']]
df_vpd = df_vpd[['FIPS', 'year','VPD-Mean']]
df_warm = df_warm[['FIPS', 'year','Thr31']]


# **Yield Analysis**

In [ ]:
cond_irig = df_yield['prodn_practice_desc'] == 'ALL PRODUCTION PRACTICES'
df_yield = df_yield[cond_irig]
df_yield['prodn_practice_desc'] = 'ALL'
df_yield = df_yield.rename(columns={'commodity_desc':'Crop','prodn_practice_desc':'Irrigation Practice'})
df_yield['FIPS'] = df_yield['FIPS'].astype(int)

In [ ]:
df_yield.iloc[0:3]

In [ ]:
df_merge_yield_ins = pd.merge(df_yield,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_yield_ins_svi = pd.merge(df_merge_yield_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_yield_ins_svi_gwu = pd.merge(df_merge_yield_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')
# df_merge_crop_ins_svi_gwu

df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu.dropna(subset='d_yield_standard')
condy = df_merge_yield_ins_svi_gwu.year < 2023
df_merge_yield_ins_svi_gwu = df_merge_yield_ins_svi_gwu[condy]

# df_merge_crop_ins_svi_gwu['Liability'] = df_merge_crop_ins_svi_gwu['Liability'].fillna(0)
# df_merge_crop_ins_svi_gwu['Subsidy'] = df_merge_crop_ins_svi_gwu['Subsidy'].fillna(0)
# df_merge_crop_ins_svi_gwu

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])
# df_merge_clim

df_merge = pd.merge(df_merge_yield_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()
df_merge

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]

df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]
df_merge

In [ ]:
Cause_of_Loss[-3]

'Drought'

In [ ]:
import pandas as pd

cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond1 = df_merge['Irrigation Practice'] == 'ALL'
df_merge_all = df_merge[cond1 & cond_loss]

drop_cols = ['FIPS', 	'Crop', 	'Irrigation Practice', 	'year' , 'Cause_of_Loss_Desc']
df_merge_all = df_merge_all.drop(columns=drop_cols)

corr_mat_all = df_merge_all.corr()

corr_all = corr_mat_all.iloc[1:].d_yield_standard
corr_all

In [ ]:
correlation_matrix = corr_mat_all
arr = correlation_matrix.values
np.fill_diagonal(arr, 0)
correlation_matrix = pd.DataFrame(arr, columns=correlation_matrix.columns, index=correlation_matrix.index)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# correlation_matrix.to_csv('<DATA_ROOT>/MachineLearning/failShare_All_Corr_Heatmap.csv')
# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap='coolwarm')
# plt.xticks(correlation_matrix.index, rotation=45, ha='right')
plt.title('Correlation Heatmap Yield & Deriving Factors - Irrigation Type: all')
# plt.savefig('<DATA_ROOT>/MachineLearning/failShare_All_Corr_Heatmap.jpg')

In [178]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]
cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-1]

cond_ir = df_merge['Irrigation Practice'] == 'ALL'
df_merge_yield = df_merge[cond_ir & cond_loss]

In [ ]:
df_merge_yield

In [180]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'SOYBEANS' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_yield.Crop == selected_crop
df = df_merge_yield[cond_crop].drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# df_scaled

In [183]:
df_scaled.to_csv('jagh.csv',index=False)

In [181]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42) #

params = {
      'learning_rate': 0.011,
      'max_depth': 6,
      'n_estimators': 750,
      'subsample': 0.6,
          }

xgb_model = xgb.XGBRegressor(**params)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)
r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)


R-squared_test: 0.3996246826999911
R-squared_train: 0.5808721104342134


In [ ]:
0.01:
R-squared_test: 0.36824183193461335
R-squared_train: 0.8665001634609111

In [ ]:
# params = {'max_depth': 7,
#           'learning_rate': 0.001,
#           'subsample': 0.8,
#           'colsample_bytree': 0.75,
#           'colsample_bylevel': 0.75,
#           'n_estimators': 100,
#           'max_leaves':100
#             }

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)


X = df_scaled.drop(columns=['d_yield_standard'])
y = df_scaled['d_yield_standard']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1,random_state=85)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)


mean_train = np.mean(y_train)
# Get predictions on the test set
baseline_predictions = np.ones(y_test.shape) * mean_train
# Compute MAE
mae_baseline = mean_absolute_error(y_test, baseline_predictions)
print("Baseline MAE is {:.2f}".format(mae_baseline))

params = {
    'max_depth': 7,
    'learning_rate': 0.01,
    'subsample': 0.6,
    'n_estimators': 500,
}


params['eval_metric'] = "mae"
num_boost_round = 999


gridsearch_params = [ learning_rate  for learning_rate in [0.2,0.1,0.06,0.03,0.01,0.005]]
# gridsearch_params = [
#     (max_depth, min_child_weight,subsample, colsample,eta)
#     for max_depth in range(9,12)
#     for min_child_weight in range(5,8)
#     for subsample in [i/10. for i in range(7,11)]
#     for colsample in [i/10. for i in range(7,11)]
#     for eta in [.3, .2, .1, .05, .01, .005]
# ]

min_mae = float("Inf")
best_params = None
# for max_depth, min_child_weight,subsample, colsample,eta in gridsearch_params:
for learning_rate in gridsearch_params:
    # print("max_depth={}, min_child_weight={}, subsample={}, colsample={}, eta={}".format(
    #     max_depth,min_child_weight,subsample, colsample,eta))
    print(learning_rate)

    # Update our parameters
    # params['max_depth'] = max_depth
    # params['min_child_weight'] = min_child_weight
    # params['subsample'] = subsample
    # params['colsample_bytree'] = colsample
    params['learning_rate'] = learning_rate

    # Run CV
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,
        metrics={'mae'},
        early_stopping_rounds=10
    )
    # Update best MAE
    mean_mae = cv_results['test-mae-mean'].min()
    boost_rounds = cv_results['test-mae-mean'].argmin()
    print("\tMAE {}".format(mean_mae))
    if mean_mae < min_mae:
      min_mae = mean_mae
      # best_params = (max_depth,min_child_weight,subsample, colsample,eta)
      best_params = (learning_rate)

print("Best params: {}, MAE: {}".format(best_params, min_mae))

# params['max_depth'] = best_params[0]
# params['min_child_weight'] = best_params[1]
# params['subsample'] = best_params[2]
# params['colsample_bytree'] = best_params[3]
params['learning_rate'] = best_params[0]

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=50
)
print("Best MAE: {:.2f} in {} rounds".format(model.best_score, model.best_iteration+1))

num_boost_round = model.best_iteration + 1
best_model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

best_model.save_model("my_model.model")

y_pred = best_model.predict(dtest)
y_pred2 = best_model.predict(dtrain)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)

In [ ]:
params['learning_rate'] = best_params

model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")],
    early_stopping_rounds=50
)
print("Best MAE: {:.2f} in {} rounds".format(model.best_score, model.best_iteration+1))

num_boost_round = model.best_iteration + 1
best_model = xgb.train(
    params,
    dtrain,
    num_boost_round=num_boost_round,
    evals=[(dtest, "Test")]
)

best_model.save_model("my_model.model")

y_pred = best_model.predict(dtest)
y_pred2 = best_model.predict(dtrain)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)

In [ ]:
params

{'max_depth': 7,
 'min_child_weight': 6,
 'eta': 0.05,
 'subsample': 0.7,
 'colsample_bytree': 0.7,
 'objective': 'reg:linear',
 'eval_metric': 'mae'}

In [ ]:
loaded_model = xgb.Booster()
loaded_model.load_model("my_model.model")
# And use it for predictions.
loaded_model.predict(dtest)

(1, 2, 3)

# **Merge Data & PreAnalysis**

In [ ]:
# df_fail , df_yield , df_svi , dfw
print(df_fail.columns)
print(df_yield.columns)
print(dfw.columns)
print(df_insure.columns)
print(df_svi.columns)

print(df_cold.columns)
print(df_drought.columns)
print(df_prcp.columns)
print(df_vpd.columns)
print(df_warm.columns)


In [ ]:
df_fail = df_fail[df_fail.fail_share > 0]


In [ ]:
df_merge_crop_ins = pd.merge(df_fail,df_insure,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:m')
df_merge_crop_ins_svi = pd.merge(df_merge_crop_ins,df_svi,on=['FIPS', 'Crop', 'year'],how='outer',validate='m:1')
df_merge_crop_ins_svi_gwu = pd.merge(df_merge_crop_ins_svi,dfw,on=['FIPS', 'year'],how='outer',validate='m:1')
# df_merge_crop_ins_svi_gwu

df_merge_crop_ins_svi_gwu = df_merge_crop_ins_svi_gwu.dropna(subset='fail_share')
condy = df_merge_crop_ins_svi_gwu.year < 2023
df_merge_crop_ins_svi_gwu = df_merge_crop_ins_svi_gwu[condy]

# df_merge_crop_ins_svi_gwu['Liability'] = df_merge_crop_ins_svi_gwu['Liability'].fillna(0)
# df_merge_crop_ins_svi_gwu['Subsidy'] = df_merge_crop_ins_svi_gwu['Subsidy'].fillna(0)
# df_merge_crop_ins_svi_gwu

df_merge_clim = pd.merge(df_cold,df_drought,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_prcp,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_vpd,on=['FIPS', 'year'])
df_merge_clim = pd.merge(df_merge_clim,df_warm,on=['FIPS', 'year'])
# df_merge_clim

df_merge = pd.merge(df_merge_crop_ins_svi_gwu,df_merge_clim,on=['FIPS', 'year'],validate='m:1')
df_merge = df_merge.dropna()
df_merge

In [ ]:
# df_merge.Cause_of_Loss_Desc.unique()
# df_merge.groupby('Cause_of_Loss_Desc').count()[['FIPS','year']].sort_values('FIPS')

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]

df_merge = df_merge[df_merge['Cause_of_Loss_Desc'].isin(Cause_of_Loss)]
df_merge

In [ ]:
Cause_of_Loss[-3]

'Drought'

In [ ]:
import pandas as pd

cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond1 = df_merge['Irrigation Practice'] == 'ALL'
df_merge_all = df_merge[cond1 & cond_loss]

cond2 = df_merge['Irrigation Practice'] == 'N'
df_merge_nir = df_merge[cond2 & cond_loss]

cond3 = df_merge['Irrigation Practice'] == 'I'
df_merge_ir = df_merge[cond3 & cond_loss]

drop_cols = ['FIPS', 	'Crop', 	'Irrigation Practice', 	'year' , 'Cause_of_Loss_Desc']
df_merge_all = df_merge_all.drop(columns=drop_cols)
df_merge_nir = df_merge_nir.drop(columns=drop_cols)
df_merge_ir = df_merge_ir.drop(columns=drop_cols)

corr_mat_all = df_merge_all.corr()
corr_mat_nir = df_merge_nir.corr()
corr_mat_ir = df_merge_ir.corr()

cond = corr_mat_all.fail_share > 0
corr_all = corr_mat_all[cond].iloc[1:].fail_share

cond = corr_mat_nir.fail_share > 0
corr_nir = corr_mat_nir[cond].iloc[1:].fail_share

cond = corr_mat_ir.fail_share > 0
corr_ir = corr_mat_ir[cond].iloc[1:].fail_share

In [ ]:
correlation_matrix = corr_mat_ir
arr = correlation_matrix.values
np.fill_diagonal(arr, 0)
correlation_matrix = pd.DataFrame(arr, columns=correlation_matrix.columns, index=correlation_matrix.index)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# correlation_matrix.to_csv('<DATA_ROOT>/MachineLearning/failShare_All_Corr_Heatmap.csv')
# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap='coolwarm')
# plt.xticks(correlation_matrix.index, rotation=45, ha='right')
plt.title('Correlation Heatmap Failure Share & Deriving Factors - Irrigation Type: Ir')
# plt.savefig('<DATA_ROOT>/MachineLearning/failShare_All_Corr_Heatmap.jpg')

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# plt.figure(figsize=(30, 8))
# plt.plot(corr_all.index, corr_all)
# plt.xticks(corr_all.index,rotation=45, ha='right',fontsize=16)
# plt.show()

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# plt.figure(figsize=(30, 8))
# plt.plot(corr_nir.index, corr_nir)
# plt.xticks(corr_nir.index,rotation=45, ha='right',fontsize=16)
# plt.show()

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# plt.figure(figsize=(30, 8))
# plt.plot(corr_ir.index, corr_ir)
# plt.xticks(corr_ir.index,rotation=45, ha='right',fontsize=16)
# plt.show()

In [ ]:
Cause_of_Loss = [
    'Insects',	'Plant Disease',	'Frost',	'Hot Wind',
    'Wildlife',	'Freeze',	'Cold Winter',	'Wind/Excess Wind',
    'Flood',	'Hail',	'Heat',	'Cold Wet Weather',	'Drought',
    'Decline in Price',	'Excess Moisture/Precipitation/Rain'
]
cond_loss = df_merge['Cause_of_Loss_Desc'] == Cause_of_Loss[-3]

cond_ir = df_merge['Irrigation Practice'] == 'ALL'
df_merge_fail = df_merge[cond_ir & cond_loss]

In [ ]:
df_merge_fail.groupby('Crop').count()

In [ ]:
df_merge_fail.iloc[0:3]

,FIPS,Crop,Irrigation Practice,year,fail_share,Cause_of_Loss_Desc,RPL_THEMES,gwu_rate,swu_rate,Thr-2,pdsi-2,dpi1,VPD-Mean,Thr31
3,1019,SOYBEANS,ALL,2009,0.000384,Drought,0.501997,0.000000,1.180180,0.0,45.0,10.488087,1.403033,35.0
8,1019,CORN,ALL,2009,0.043390,Drought,0.510229,0.000000,1.180180,0.0,45.0,10.488087,1.403033,35.0
14,1033,CORN,ALL,2009,0.000828,Drought,0.549850,0.356061,0.530303,0.0,120.0,11.144219,1.403033,41.0


**Regression**

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

selected_crop = 'CORN' # 'WHEAT' , 'CORN' , 'SOYBEANS'
cond_crop = df_merge_fail.Crop == selected_crop
df = df_merge_fail.drop(columns=['FIPS','Crop','Irrigation Practice','year','Cause_of_Loss_Desc'])
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df_scaled

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_scaled.drop(columns=['fail_share'])
y = df_scaled['fail_share']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) #,random_state=85

# params = {'max_depth': 8,
#           'learning_rate': 0.001,
#           'subsample': 0.6,
#           'colsample_bytree': 0.6,
#           'colsample_bylevel': 0.6,
#           'n_estimators': 750
#           }

params = {'max_depth': 7,
          'learning_rate': 0.001,
          'subsample': 0.8,
          'colsample_bytree': 0.75,
          'colsample_bylevel': 0.75,
          'n_estimators': 100,
          'max_leaves':100
            }

xgb_model = xgb.XGBRegressor(**params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

r2 = r2_score(y_test, y_pred)
print("R-squared_test:", r2)

r2 = r2_score(y_train, y_pred2)
print("R-squared_train:", r2)


R-squared_test: 0.03847643369023612
R-squared_train: 0.05119614271286388


In [ ]:
import numpy as np
bins = list(dff.fail_share.quantile(np.arange(0,1,0.34)))
bins = bins + [1]
labels = list(np.arange(0,3))
dff['fail_cls'] = dff.fail_share.to_frame().apply(lambda x: pd.cut(x, bins=bins, labels=labels), axis=0)
dff['fail_cls'] = dff['fail_cls'].fillna(0)


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = dff.drop(columns=['FIPS','Crop','Irrigation Practice','year','fail_share','fail_cls'])
scaler = MinMaxScaler()
scaler.fit(df)
df_scaled = pd.DataFrame(scaler.transform(df), columns=df.columns)


In [ ]:
random_state_SOYBEANS = 85
params_SOYBEANS = {'max_depth': 7,
                  'learning_rate': 0.055,
                  'subsample': 0.65,
                  'colsample_bytree': 0.75,
                  'colsample_bylevel': 0.75,
                  'n_estimators': 650,
                  'max_leaves':100,
                  }

random_state_CORN = 85
params_CORN = {'max_depth': 7,
              'learning_rate': 0.05,
              'subsample': 0.8,
              'colsample_bytree': 0.75,
              'colsample_bylevel': 0.75,
              'n_estimators': 650,
              'max_leaves':100
               }

random_state_WHEAT = 100
params_WHEAT = {'max_depth': 7,
                'learning_rate': 0.007,
                'subsample': 0.8,
                'colsample_bytree': 0.55,
                'colsample_bylevel': 0.7,
                'n_estimators': 650,
                'max_leaves':100,
              }


In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df_scaled
y = dff['fail_cls']

if selected_crop == 'SOYBEANS':
  random_state = random_state_SOYBEANS
  params = params_SOYBEANS
elif selected_crop == 'CORN':
  random_state = random_state_CORN
  params = params_CORN
elif selected_crop == 'WHEAT':
  random_state = random_state_WHEAT
  params = params_WHEAT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1,random_state=random_state)

xgb_model = xgb.XGBClassifier(**params)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_pred2 = xgb_model.predict(X_train)

accuracy_test = accuracy_score(y_test, y_pred)
print("Accuracy_test: %.2f%%" % (accuracy_test * 100.0))
accuracy_train = accuracy_score(y_train, y_pred2)
print("Accuracy_train: %.2f%%" % (accuracy_train * 100.0))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(np.array([cm[0,0],cm[1,1],cm[2,2]])/np.sum(cm,axis=1))